# Analysis: LLMs (via Ollama) against our engines

Reads the results produced by `scripts/run_llm_validation.py` -- run on a remote server,
not here -- and applies the same kind of statistical analysis as
`notebooks/validation_exhaustive.ipynb` (bootstrap CI, paired comparison with Wilcoxon +
Holm-Bonferroni), plus the metrics specific to this experiment: structured-output
compliance rate and latency per model.

**Experiment design** (see `scripts/run_llm_validation.py`): each LLM is embedded in a
small but real market -- 1 LLM instance + 3 rival engines on its own side (R as the
floor, OPT2 as the best hand-designed engine, ML2 as the best learned engine), against a
fixed roster of 4 engines on the other side (R, RV, OPT2, IS) -- for a configurable number
of days. The LLM receives the complete decision rules (exact caps and floors) and the
full memory of its own table (what it has offered, what it has been offered), as designed.

This notebook does NOT run any simulation -- it only reads and analyses. If
`data/llm_validation/` is empty, first run `python scripts/run_llm_validation.py` (see
`paper/README.md`).


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT.parent))  # repo root: the wehaveadeal package

import json
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests

from wehaveadeal.stats import bootstrap_ci, paired_wilcoxon
from wehaveadeal.llm.roster import tier_of, family_of, TIER_ORDER, FAMILY_ORDER

sns.set_theme(style="whitegrid")

DATA_DIR = PROJECT_ROOT / "data" / "llm_validation"
outcome_files = sorted(DATA_DIR.glob("*_outcomes.jsonl"))
calls_files = sorted(DATA_DIR.glob("*_calls.jsonl"))
print(f"{len(outcome_files)} outcome files, {len(calls_files)} calls files in {DATA_DIR}")
if not outcome_files:
    print("No results yet -- run `python scripts/run_llm_validation.py` first (see paper/README.md).")


18 outcome files, 18 calls files in C:\Users\USER\Documents\we-have-a-deal\data\llm_validation


## Cargar resultados

In [2]:
def load_jsonl(paths):
    rows = []
    for p in paths:
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    return pd.DataFrame(rows)

raw = load_jsonl(outcome_files)
calls = load_jsonl(calls_files)

if len(raw) > 0:
    raw = raw[raw["error"].isna()].copy() if "error" in raw.columns else raw  # drop failed-day placeholder rows
    raw["label"] = raw.apply(lambda r: r["model"] if r["is_llm"] else r["agent_id"].rsplit("-", 2)[1], axis=1)
    raw["day_id"] = raw["model"].astype(str) + "|" + raw["role"] + "|" + raw["day_index"].astype(str)
    # Older runs predate the tier/family columns the runner now stamps -- derive them from
    # the roster instead of re-running, so old and new JSONL can be analyzed together.
    if "tier" not in raw.columns:
        raw["tier"] = raw["model"].map(tier_of)
    if "family" not in raw.columns:
        raw["family"] = raw["model"].map(family_of)
    raw["tier"] = pd.Categorical(raw["tier"], categories=TIER_ORDER, ordered=True)
    raw["family"] = pd.Categorical(raw["family"], categories=FAMILY_ORDER, ordered=True)

if len(calls) > 0:
    if "tier" not in calls.columns:
        calls["tier"] = calls["model"].map(tier_of)
    if "family" not in calls.columns:
        calls["family"] = calls["model"].map(family_of)
    calls["tier"] = pd.Categorical(calls["tier"], categories=TIER_ORDER, ordered=True)
    calls["family"] = pd.Categorical(calls["family"], categories=FAMILY_ORDER, ordered=True)

print(f"{len(raw)} valid outcome rows, {len(calls)} model-call log rows")
raw.head()


2828 valid outcome rows, 7535 model-call log rows


,model,tier,family,role,day_index,seed,agent_id,is_llm,matched,outcome,elapsed_s,error,label,day_id
0,deepseek-r1:1.5b,tiny,deepseek-r1,carrier,0,90924400,carrier-llm-1,True,True,-74.809298,272.062,NaN,deepseek-r1:1.5b,deepseek-r1:1.5b|carrier|0
1,deepseek-r1:1.5b,tiny,deepseek-r1,carrier,0,90924400,carrier-r-1,False,True,-140.150843,272.062,NaN,r,deepseek-r1:1.5b|carrier|0
2,deepseek-r1:1.5b,tiny,deepseek-r1,carrier,0,90924400,carrier-opt2-1,False,True,85.283454,272.062,NaN,opt2,deepseek-r1:1.5b|carrier|0
3,deepseek-r1:1.5b,tiny,deepseek-r1,carrier,0,90924400,carrier-ml2-1,False,True,68.385880,272.062,NaN,ml2,deepseek-r1:1.5b|carrier|0
4,deepseek-r1:1.5b,tiny,deepseek-r1,carrier,1,90924401,carrier-llm-1,True,True,-200.392446,213.438,NaN,deepseek-r1:1.5b,deepseek-r1:1.5b|carrier|1


## Structured-output compliance and latency per model

How many calls produced valid JSON on the first attempt, how many needed retries, and how
long each model took to respond. This is a reportable result in its own right -- not just
a technical detail.


In [3]:
if len(calls) > 0:
    calls["is_retry_or_forfeit"] = calls["attempt"] != 0
    compliance = calls.groupby(["tier", "family", "model"], observed=True).agg(
        n_calls=("model", "size"),
        first_try_ok_rate=("attempt", lambda s: (s == 0).mean()),
        forfeit_rate=("attempt", lambda s: (s == "forfeit").mean()),
        mean_latency_s=("latency_s", "mean"),
        median_latency_s=("latency_s", "median"),
        p95_latency_s=("latency_s", lambda s: s.quantile(0.95)),
    ).sort_index()
    display(compliance)
else:
    print("no call logs yet")


n_calls  first_try_ok_rate  forfeit_rate  \
tier   family      model                                                        
tiny   llama       llama3.2:1b          1158           0.286701      0.208981   
       qwen        qwen2.5:1.5b          694           0.512968      0.134006   
       deepseek-r1 deepseek-r1:1.5b      766           0.506527      0.122715   
small  llama       llama3.1:8b           497           0.883300      0.028169   
       qwen        qwen2.5:7b            837           0.707288      0.057348   
       deepseek-r1 deepseek-r1:7b        631           0.705230      0.077655   
medium llama       llama3.3:70b          837           0.847073      0.046595   
       qwen        qwen2.5:72b           830           0.842169      0.015663   
       deepseek-r1 deepseek-r1:70b      1285           0.253696      0.245136   

                                     mean_latency_s  median_latency_s  \
tier   family      model                                                
tiny   llama       llama3.2:1b             2.267362            2.2500   
       qwen        qwen2.5:1.5b            2.300481            2.3120   
       deepseek-r1 deepseek-r1:1.5b       19.537935            4.9375   
small  llama       llama3.1:8b             2.570752            2.5620   
       qwen        qwen2.5:7b              2.409181            2.3910   
       deepseek-r1 deepseek-r1:7b          9.793268            6.7030   
medium llama       llama3.3:70b           29.811497           29.5235   
       qwen        qwen2.5:72b            37.022108           33.1410   
       deepseek-r1 deepseek-r1:70b        91.797040           92.0620   

                                     p95_latency_s  
tier   family      model                            
tiny   llama       llama3.2:1b             2.43700  
       qwen        qwen2.5:1.5b            2.42200  
       deepseek-r1 deepseek-r1:1.5b       92.06200  
small  llama       llama3.1:8b             2.68700  
       qwen        qwen2.5:7b              2.54700  
       deepseek-r1 deepseek-r1:7b         18.67120  
medium llama       llama3.3:70b           37.99415  
       qwen        qwen2.5:72b            64.13740  
       deepseek-r1 deepseek-r1:70b        92.09300

## Descriptive statistics per model/rival, with bootstrap CIs

In [4]:
summary_rows = []
if len(raw) > 0:
    for (role, label), g in raw.groupby(["role", "label"]):
        mean, lo, hi = bootstrap_ci(g["outcome"].to_numpy())
        summary_rows.append({
            "role": role, "label": label, "n": len(g),
            "match_rate": g["matched"].mean(), "mean_outcome": mean, "ci_lo": lo, "ci_hi": hi,
        })
summary = pd.DataFrame(summary_rows)
if len(summary):
    summary = summary.sort_values(["role", "mean_outcome"], ascending=[True, False])
if len(summary):
    summary.to_csv(DATA_DIR / "summary_stats.csv", index=False)
summary


,role,label,n,match_rate,mean_outcome,ci_lo,ci_hi
6,carrier,ml2,351,1.000000,68.636936,6.649730e+01,70.810860
7,carrier,opt2,351,1.000000,63.897843,6.149975e+01,66.254305
5,carrier,llama3.3:70b,40,0.825000,25.389713,5.635583e+00,43.233145
1,carrier,deepseek-r1:70b,40,0.025000,-0.436877,-1.310632e+00,0.000000
4,carrier,llama3.2:1b,40,0.375000,-18.073677,-3.319129e+01,-3.756841
9,carrier,qwen2.5:72b,40,1.000000,-20.853438,-4.445263e+01,0.884231
11,carrier,r,351,1.000000,-60.941612,-6.875362e+01,-53.225073
8,carrier,qwen2.5:1.5b,40,0.800000,-63.551182,-8.810931e+01,-40.689761
0,carrier,deepseek-r1:1.5b,40,0.900000,-64.577792,-8.970522e+01,-38.988773
2,carrier,deepseek-r1:7b,36,0.972222,-87.648430,-1.143197e+02,-59.991203


## Visualisation: each LLM against its 3 fixed rivals (R, OPT2, ML2), by role

In [ ]:
if len(summary):
    roles = summary["role"].unique()
    fig, axes = plt.subplots(1, len(roles), figsize=(7 * len(roles), 5), squeeze=False)
    for i, role in enumerate(roles):
        ax = axes[0][i]
        sub = summary[summary.role == role].sort_values("mean_outcome", ascending=False)
        colors = ["#4a3aa7" if lbl in ("r", "opt2", "ml2") else "#eb6834" for lbl in sub["label"]]
        ax.barh(sub["label"], sub["mean_outcome"], color=colors,
                xerr=[sub["mean_outcome"] - sub["ci_lo"], sub["ci_hi"] - sub["mean_outcome"]], capsize=3)
        ax.set_title(f"{role}  (purple = our engine, orange = LLM)")
        ax.axvline(0, color="black", linewidth=0.8)
    plt.tight_layout()
    plt.savefig(DATA_DIR / "llm_vs_engines.png", dpi=150)
    plt.show()

## Aggregate view by tier and family (LLM only)

Collapses the 3 fixed engine rivals (R/OPT2/ML2) and looks only at the LLM agent, grouped by
size tier (tiny <3B / small 4-14B / medium 15-70B) and family (Llama, Qwen, DeepSeek R1
distill). Bootstrap CI as in the rest of the notebook -- **not** paired across models:
each (model, role) runs over its own independent sequence of days/seeds (see `combo_seed`
in `scripts/run_llm_validation.py`), so this is a descriptive comparison, not a paired
hypothesis test like the section below.

In [6]:
tier_rows, tier_family_rows = [], []
if len(raw) > 0:
    llm_rows = raw[raw.is_llm]
    for (role, tier), g in llm_rows.groupby(["role", "tier"], observed=True):
        mean, lo, hi = bootstrap_ci(g["outcome"].to_numpy())
        tier_rows.append({"role": role, "tier": tier, "n": len(g), "match_rate": g["matched"].mean(),
                           "mean_outcome": mean, "ci_lo": lo, "ci_hi": hi})
    for (role, tier, family), g in llm_rows.groupby(["role", "tier", "family"], observed=True):
        mean, lo, hi = bootstrap_ci(g["outcome"].to_numpy())
        tier_family_rows.append({"role": role, "tier": tier, "family": family, "n": len(g),
                                  "match_rate": g["matched"].mean(), "mean_outcome": mean, "ci_lo": lo, "ci_hi": hi})

tier_summary = pd.DataFrame(tier_rows)
tier_family_summary = pd.DataFrame(tier_family_rows)
for df in (tier_summary, tier_family_summary):
    if len(df):
        df["tier"] = pd.Categorical(df["tier"], categories=TIER_ORDER, ordered=True)
        if "family" in df.columns:
            df["family"] = pd.Categorical(df["family"], categories=FAMILY_ORDER, ordered=True)
if len(tier_summary):
    tier_summary = tier_summary.sort_values(["role", "tier"])
    tier_summary.to_csv(DATA_DIR / "tier_summary.csv", index=False)
if len(tier_family_summary):
    tier_family_summary = tier_family_summary.sort_values(["role", "tier", "family"])
    tier_family_summary.to_csv(DATA_DIR / "tier_family_summary.csv", index=False)

display(tier_summary)
display(tier_family_summary)


,role,tier,n,match_rate,mean_outcome,ci_lo,ci_hi
0,carrier,tiny,120,0.691667,-48.734217,-61.854888,-36.260422
1,carrier,small,111,0.990991,-112.646981,-126.413506,-98.569446
2,carrier,medium,120,0.616667,1.366466,-9.324846,11.602497
3,shipper,tiny,120,0.808333,281.665569,196.031512,373.430178
4,shipper,small,117,0.863248,511.664446,406.558338,617.777219
5,shipper,medium,119,0.663866,721.022894,596.532946,848.592273


,role,tier,family,n,match_rate,mean_outcome,ci_lo,ci_hi
0,carrier,tiny,llama,40,0.375000,-18.073677,-3.319129e+01,-3.756841
1,carrier,tiny,qwen,40,0.800000,-63.551182,-8.810931e+01,-40.689761
2,carrier,tiny,deepseek-r1,40,0.900000,-64.577792,-8.970522e+01,-38.988773
3,carrier,small,llama,37,1.000000,-112.978079,-1.340590e+02,-90.714976
4,carrier,small,qwen,38,1.000000,-136.007432,-1.574482e+02,-112.460762
5,carrier,small,deepseek-r1,36,0.972222,-87.648430,-1.143197e+02,-59.991203
6,carrier,medium,llama,40,0.825000,25.389713,5.635583e+00,43.233145
7,carrier,medium,qwen,40,1.000000,-20.853438,-4.445263e+01,0.884231
8,carrier,medium,deepseek-r1,40,0.025000,-0.436877,-1.310632e+00,0.000000
9,shipper,tiny,llama,40,0.500000,2.667750,1.728040e-12,7.828250


## Visualisation: mean LLM outcome by tier, broken down by family

In [ ]:
if len(tier_family_summary):
    roles = tier_family_summary["role"].unique()
    fig, axes = plt.subplots(1, len(roles), figsize=(6.5 * len(roles), 5), squeeze=False, sharey=True)
    family_colors = {"llama": "#4a3aa7", "qwen": "#eb6834", "deepseek-r1": "#1b9e91"}
    bar_width = 0.25
    x_base = np.arange(len(TIER_ORDER))
    for i, role in enumerate(roles):
        ax = axes[0][i]
        sub = tier_family_summary[tier_family_summary.role == role]
        for j, family in enumerate(FAMILY_ORDER):
            fam_sub = sub[sub.family == family].set_index("tier").reindex(TIER_ORDER)
            x = x_base + (j - 1) * bar_width
            yerr = [fam_sub["mean_outcome"] - fam_sub["ci_lo"], fam_sub["ci_hi"] - fam_sub["mean_outcome"]]
            ax.bar(x, fam_sub["mean_outcome"], width=bar_width, yerr=yerr, capsize=3,
                   color=family_colors[family], label=family)
        ax.set_xticks(x_base)
        ax.set_xticklabels(TIER_ORDER)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.set_title(role)
        ax.legend(title="family")
    plt.tight_layout()
    plt.savefig(DATA_DIR / "llm_by_tier_family.png", dpi=150)
    plt.show()

## Paired comparison: each LLM against each of its 3 fixed rivals (same day)

The 4 agents (LLM + R + OPT2 + ML2) always share the same market on the same day -- a
fully paired design, with no need to filter by co-presence (unlike the scarce conditions
of the other notebook). We apply Holm-Bonferroni to the 3 comparisons of each
(model, role).


In [8]:
pairwise_rows = []
if len(raw) > 0:
    for role in raw["role"].unique():
        for model in raw[raw.is_llm]["model"].unique():
            sub = raw[(raw.role == role) & ((raw.model == model))]
            wide = sub.pivot_table(index="day_id", columns="label", values="outcome")
            if model not in wide.columns:
                continue
            for rival in ["r", "opt2", "ml2"]:
                if rival not in wide.columns:
                    continue
                pair = wide[[model, rival]].dropna()
                if len(pair) < 5:
                    continue
                mean_diff, ci_lo, ci_hi, p_raw = paired_wilcoxon(pair[model], pair[rival])
                pairwise_rows.append({
                    "role": role, "model": model, "rival": rival, "n_days": len(pair),
                    "mean_diff": mean_diff, "ci_lo": ci_lo, "ci_hi": ci_hi, "p_raw": p_raw,
                })

pairwise = pd.DataFrame(pairwise_rows)
if len(pairwise):
    pairwise["tier"] = pd.Categorical(pairwise["model"].map(tier_of), categories=TIER_ORDER, ordered=True)
    pairwise["family"] = pd.Categorical(pairwise["model"].map(family_of), categories=FAMILY_ORDER, ordered=True)
    pairwise["p_holm"] = np.nan
    pairwise["significant"] = False
    for (role, model), idx in pairwise.groupby(["role", "model"]).groups.items():
        reject, p_corrected, _, _ = multipletests(pairwise.loc[idx, "p_raw"], alpha=0.05, method="holm")
        pairwise.loc[idx, "p_holm"] = p_corrected
        pairwise.loc[idx, "significant"] = reject
    pairwise = pairwise.sort_values(["role", "tier", "family", "rival"])
    pairwise.to_csv(DATA_DIR / "pairwise_vs_rivals.csv", index=False)
pairwise


,role,model,rival,n_days,mean_diff,ci_lo,ci_hi,p_raw,tier,family,p_holm,significant
14,carrier,llama3.2:1b,ml2,40,-89.375764,-105.595275,-74.004978,5.456968e-12,tiny,llama,1.637090e-11,True
13,carrier,llama3.2:1b,opt2,40,-84.997310,-101.922805,-69.133143,3.456080e-11,tiny,llama,6.912160e-11,True
12,carrier,llama3.2:1b,r,40,48.836932,20.846064,75.692695,3.867511e-03,tiny,llama,3.867511e-03,True
20,carrier,qwen2.5:1.5b,ml2,40,-134.236692,-159.703683,-107.979005,3.637979e-12,tiny,qwen,7.275958e-12,True
19,carrier,qwen2.5:1.5b,opt2,40,-129.495533,-153.792795,-104.556283,1.818989e-12,tiny,qwen,5.456968e-12,True
18,carrier,qwen2.5:1.5b,r,40,-9.534997,-38.409396,20.161806,6.558993e-01,tiny,qwen,6.558993e-01,False
2,carrier,deepseek-r1:1.5b,ml2,40,-137.019513,-161.492205,-112.689183,1.273293e-11,tiny,deepseek-r1,3.819878e-11,True
1,carrier,deepseek-r1:1.5b,opt2,40,-126.100956,-150.760303,-100.796619,1.600711e-10,tiny,deepseek-r1,3.201421e-10,True
0,carrier,deepseek-r1:1.5b,r,40,5.334948,-23.686151,35.827058,8.995151e-01,tiny,deepseek-r1,8.995151e-01,False
11,carrier,llama3.1:8b,ml2,37,-177.219111,-201.964263,-152.154478,1.455192e-11,small,llama,4.365575e-11,True


## Direct reading: do our engines beat the LLM?

mean_diff = LLM - rival, so a negative, significant value means the LLM lost to that
rival engine on average over the same days; a positive one means the LLM beat it.


In [9]:
if len(pairwise):
    for _, row in pairwise.sort_values(["role", "tier", "family", "rival"]).iterrows():
        verdict = "SIGNIFICATIVO" if row["significant"] else "no concluyente"
        who = "LLM le gana a" if row["mean_diff"] > 0 else "LLM pierde contra"
        print(f"[{row['role']:8s}] {row['tier']:6s}/{row['family']:12s} {row['model']:16s} vs {row['rival']:5s}: "
              f"{who} (diff={row['mean_diff']:+.2f}, IC=[{row['ci_lo']:+.2f}, {row['ci_hi']:+.2f}], "
              f"p_holm={row['p_holm']:.4f}) -- {verdict}")


[carrier ] tiny  /llama        llama3.2:1b      vs ml2  : LLM pierde contra (diff=-89.38, IC=[-105.60, -74.00], p_holm=0.0000) -- SIGNIFICATIVO
[carrier ] tiny  /llama        llama3.2:1b      vs opt2 : LLM pierde contra (diff=-85.00, IC=[-101.92, -69.13], p_holm=0.0000) -- SIGNIFICATIVO
[carrier ] tiny  /llama        llama3.2:1b      vs r    : LLM le gana a (diff=+48.84, IC=[+20.85, +75.69], p_holm=0.0039) -- SIGNIFICATIVO
[carrier ] tiny  /qwen         qwen2.5:1.5b     vs ml2  : LLM pierde contra (diff=-134.24, IC=[-159.70, -107.98], p_holm=0.0000) -- SIGNIFICATIVO
[carrier ] tiny  /qwen         qwen2.5:1.5b     vs opt2 : LLM pierde contra (diff=-129.50, IC=[-153.79, -104.56], p_holm=0.0000) -- SIGNIFICATIVO
[carrier ] tiny  /qwen         qwen2.5:1.5b     vs r    : LLM pierde contra (diff=-9.53, IC=[-38.41, +20.16], p_holm=0.6559) -- no concluyente
[carrier ] tiny  /deepseek-r1  deepseek-r1:1.5b vs ml2  : LLM pierde contra (diff=-137.02, IC=[-161.49, -112.69], p_holm=0.0000) -- SIGNIF